Environment Setup & Cloud Authentication

In [1]:
import json
import pandas as pd
from azure.storage.blob import BlobServiceClient

ACCOUNT_URL = "https://aegisscholardata.blob.core.windows.net"
SAS_TOKEN = "?sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupiytfx&se=2026-05-30T08:31:15Z&st=2026-02-12T01:16:15Z&spr=https&sig=pU5C5a%2B%2BxE1zvMMv3vjUqjlJXC9dMgpsVyM0V%2FfuEIo%3D"
CONTAINER = "raw"
PREFIX = "dtic/works/" 

client = BlobServiceClient(account_url=ACCOUNT_URL, credential=SAS_TOKEN)
container = client.get_container_client(CONTAINER)
print("Azure Connection Ready.")

Azure Connection Ready.


Data Extraction & Relational Mapping
Result is stored as a "Long-Form" DataFrame (master_edge_df)

In [2]:
raw_edges = []
blobs = container.list_blobs(name_starts_with=PREFIX)
processed_count = 0

for blob in blobs:
    if not blob.name.endswith('.json'):
        continue
    try:
        blob_client = container.get_blob_client(blob.name)
        record = json.loads(blob_client.download_blob().readall())
        
        paper_id = record.get('publication_id')
        
        org_lookup = {
            org.get('name'): org.get('org_id') 
            for org in record.get('organizations', []) 
            if org.get('name')
        }
        
        topics = []
        for kw_group in record.get('keywords', []):
            for entity in kw_group.get('entities', []):
                name = entity.get('details', {}).get('name')
                if name: topics.append(name)
        
        for auth in record.get('authors', []):
            auth_name = auth.get('name')
            auth_id = auth.get('researcher_id')
            affils = auth.get('affiliations', [])
            
            org_name = affils[0] if affils else "Unknown"
            grid_id = org_lookup.get(org_name, "Unknown")
            
            for topic in topics:
                raw_edges.append({
                    'author_id': auth_id,
                    'author_name': auth_name,
                    'organization': org_name,
                    'grid_id': grid_id,
                    'topic': topic,
                    'paper_id': paper_id
                })
        
        processed_count += 1
        if processed_count % 500 == 0:
            print(f"Processed {processed_count} files...")
    except:
        continue

master_edge_df = pd.DataFrame(raw_edges)
print(f"Done! {len(master_edge_df):,} connections loaded.")

Processed 500 files...
Processed 1000 files...
Processed 1500 files...
Processed 2000 files...
Processed 2500 files...
Processed 3000 files...
Processed 3500 files...
Processed 4000 files...
Processed 4500 files...
Processed 5000 files...
Processed 5500 files...
Processed 6000 files...
Processed 6500 files...
Processed 7000 files...
Processed 7500 files...
Processed 8000 files...
Processed 8500 files...
Processed 9000 files...
Processed 9500 files...
Processed 10000 files...
Processed 10500 files...
Processed 11000 files...
Processed 11500 files...
Processed 12000 files...
Processed 12500 files...
Processed 13000 files...
Processed 13500 files...
Processed 14000 files...
Processed 14500 files...
Processed 15000 files...
Processed 15500 files...
Processed 16000 files...
Processed 16500 files...
Processed 17000 files...
Processed 17500 files...
Processed 18000 files...
Processed 18500 files...
Processed 19000 files...
Processed 19500 files...
Processed 20000 files...
Processed 20500 file

Aggregates the raw connections to create a Bipartite Edge List
Output: DTIC_Author_Topic_Investigation.csv

In [3]:
investigation_df = master_edge_df.groupby(
    ['author_id', 'author_name', 'organization', 'grid_id', 'topic']
).agg(
    paper_count=('paper_id', 'nunique')
).reset_index()

investigation_df = investigation_df.sort_values(by=['topic', 'paper_count'], ascending=[True, False])

output_csv = "DTIC_Author_Topic_Investigation.csv"
investigation_df.to_csv(output_csv, index=False)

print(f"Created: {output_csv}")
display(investigation_df.head(15))

Created: DTIC_Author_Topic_Investigation.csv


,author_id,author_name,organization,grid_id,topic,paper_count
140303,ur.01305603266.08,V. R. Beasley,University of Illinois at Urbana-Champaign,grid.35403.31,"30 Agricultural, Veterinary and Food Sciences",4
161578,ur.01343700552.69,W. M. Haschek,University of Illinois at Urbana-Champaign,grid.35403.31,"30 Agricultural, Veterinary and Food Sciences",4
38293,ur.01065272330.24,George P. Naughton,University of Idaho,grid.266456.5,"30 Agricultural, Veterinary and Food Sciences",3
86773,ur.01172510264.03,W. W. Carmichael,University of Illinois at Urbana-Champaign,grid.35403.31,"30 Agricultural, Veterinary and Food Sciences",3
134032,ur.012753267537.53,Richard S. Brown,Pacific Northwest National Laboratory,grid.451303.0,"30 Agricultural, Veterinary and Food Sciences",3
175422,ur.01370035233.98,S. B. Hooser,University of Illinois at Urbana-Champaign,grid.35403.31,"30 Agricultural, Veterinary and Food Sciences",3
287,ur.010001422101.46,D. D. Muir,Flour Milling and Baking Research Association,grid.423346.4,"30 Agricultural, Veterinary and Food Sciences",2
6555,ur.01010651066.70,Changjiang Huang,Louisiana State University Agricultural Center,grid.250060.1,"30 Agricultural, Veterinary and Food Sciences",2
41308,ur.01071300673.75,Matthew L. Keefer,University of Idaho,grid.266456.5,"30 Agricultural, Veterinary and Food Sciences",2
58074,ur.011216467751.49,Ryan A. Harnish,Pacific Northwest National Laboratory,grid.451303.0,"30 Agricultural, Veterinary and Food Sciences",2


Author Skill-Profile Pivot & Classification Matrix
Output: Author_Skill_Profiles.csv

In [4]:
skill_matrix = investigation_df.pivot_table(
    index=['author_id', 'author_name', 'grid_id'], 
    columns='topic', 
    values='paper_count', 
    aggfunc='sum',
    fill_value=0
)

skill_matrix['Skill_Breadth'] = (skill_matrix > 0).sum(axis=1)
skill_matrix['Total_Depth'] = skill_matrix.iloc[:, :-1].sum(axis=1)
skill_matrix = skill_matrix.sort_values(by='Total_Depth', ascending=False)

skill_matrix.to_csv("Author_Skill_Profiles.csv")

print("Profiles Generated!")
display(skill_matrix.head(15))

Profiles Generated!


,,topic,"30 Agricultural, Veterinary and Food Sciences",3001 Agricultural Biotechnology,"3002 Agriculture, Land and Farm Management",3003 Animal Production,3004 Crop and Pasture Production,3005 Fisheries Sciences,3006 Food Sciences,3007 Forestry Sciences,3008 Horticultural Production,3009 Veterinary Sciences,...,5109 Space Sciences,5110 Synchrotrons and Accelerators,52 Psychology,5201 Applied and Developmental Psychology,5202 Biological Psychology,5203 Clinical and Health Psychology,5204 Cognitive and Computational Psychology,5205 Social and Personality Psychology,Skill_Breadth,Total_Depth
author_id,author_name,grid_id,,,,,,,,,,,,,,,,,,,,,
ur.011513406716.58,G. Lucovsky,grid.40803.3f,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,9,152
ur.013377525417.83,Rolf D. Reitz,grid.14003.36,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,107
ur.015041470323.25,Ronald D. Schrimpf,grid.152326.1,0,0,0,0,0,0,0,0,0,0,...,0,8,0,0,0,0,0,0,8,91
ur.013204260503.45,John D. Cressler,grid.213917.f,0,0,0,0,0,0,0,0,0,0,...,0,6,0,0,0,0,0,0,5,86
ur.015041470323.25,R. D. Schrimpf,grid.152326.1,0,0,0,0,0,0,0,0,0,0,...,0,3,0,0,0,0,0,0,6,83
ur.015276270062.44,S. J. Pearton,grid.15276.37,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12,82
ur.012574062215.75,George J. Pappas,grid.25879.31,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,16,76
ur.011616623171.15,Paul W. Marshall,grid.133275.1,0,0,0,0,0,0,0,0,0,0,...,0,6,0,0,0,0,0,0,6,69
ur.07427716771.40,D. M. Fleetwood,grid.152326.1,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,5,67
